<a href="https://colab.research.google.com/github/sudo-ansh007/forecast/blob/main/win_probability_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deal Win Probability — forecast_v0

Trains on closed deals, outputs a calibrated win probability per open deal.

**Upload two files:** `ml_train.csv` (3,573 closed deals, labelled) and
`ml_score.csv` (1,912 open deals, unlabelled). Nothing else needed —
this notebook is self-contained.

---

### ⚠️ Before you upload

These CSVs hold **real customer CRM data** — deal values, deal IDs, segments.
Colab is Google-hosted and notebooks are shareable by link. Clear it with
whoever owns data governance, or run this locally in Jupyter instead
(identical code, no third-party exposure).

Never paste an API token into a cell — notebooks save their output.

## 1. Setup

In [ ]:
import sys, warnings
warnings.filterwarnings("ignore")   # sigmoid calibrator overflows harmlessly at a 15.1% base rate

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingClassifier
# XGBoost is the shipping model as of section 4c. LightGBM is still imported -- it is
# the previous shipping model and stays in the benchmark. Colab has libomp already;
# locally you may need `brew install libomp` on a Mac.
try:
    from lightgbm import LGBMClassifier
    from xgboost import XGBClassifier
except ImportError:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lightgbm", "xgboost"],
                   check=True)
    from lightgbm import LGBMClassifier
    from xgboost import XGBClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.inspection import permutation_importance
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (average_precision_score, brier_score_loss,
                             roc_auc_score)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
IN_COLAB = "google.colab" in sys.modules

# Bump this when the dataset contract changes. Cell 4 refuses to run on CSVs that do not
# match, so a stale upload fails loudly instead of quietly producing different numbers.
NOTEBOOK_VERSION = "v3-2026-08-11-xgboost-evidence-gate"
print(f"notebook {NOTEBOOK_VERSION}")
print(f"colab={IN_COLAB}  pandas={pd.__version__}")

In [ ]:
import os

# Colab's /content SURVIVES a runtime restart, so a bare os.path.exists() check finds
# the PREVIOUS run's CSVs and skips the upload -- the notebook then quietly scores the
# old dataset. Check the shape, not just the filename: any mismatch deletes the stale
# files and re-prompts. Set FORCE_UPLOAD = True to re-upload unconditionally.
FORCE_UPLOAD = False

def _stale():
    """True if the local CSVs are missing or are not the v3 dataset."""
    if FORCE_UPLOAD or not os.path.exists("ml_train.csv") or not os.path.exists("ml_score.csv"):
        return True
    t, s = pd.read_csv("ml_train.csv", nrows=None), pd.read_csv("ml_score.csv", nrows=1)
    return len(t) != 3573 or "evidence" not in s.columns

if IN_COLAB and _stale():
    for f in ("ml_train.csv", "ml_score.csv"):
        if os.path.exists(f):
            os.remove(f)
            print(f"removed stale {f}")
    from google.colab import files
    print(f"Upload ml_train.csv and ml_score.csv for {NOTEBOOK_VERSION}")
    files.upload()

# Sort by created_date NOW so every split below is a time split. CSV row order is
# whatever the source table returned -- a positional split on it is quasi-random and
# inflates PR-AUC from 0.56 to 0.75 by putting a deal's future neighbours in train.
train = pd.read_csv("ml_train.csv").sort_values("created_date").reset_index(drop=True)
score = pd.read_csv("ml_score.csv")

FEATURES = ["days_in_current_stage", "days_in_sales_cycle", "stall_ratio",
            "contract_months", "num_stakeholders", "has_champion", "poc",
            "opportunity_type", "source", "geo", "segment"]
CATEGORICALS = ["opportunity_type", "source", "geo", "segment"]
LABEL = "is_won"

# display_id, acv and created_date ride along for joining / revenue weighting /
# splitting -- NOT features. acv is filled after signing (93% of won deals vs 29%
# of lost) and created_date carries cohort censoring, so either would leak.
assert set(FEATURES).issubset(train.columns), set(FEATURES) - set(train.columns)

# ---- VERSION GUARD -------------------------------------------------------------------
# The dataset changed shape twice: DROP_LOST_WITHOUT_AMOUNT cut train from 11,051 to
# 3,573 rows (base rate 4.9% -> 15.1%), and `evidence` was added to gate the forecast.
# Running this notebook on a pre-filter CSV produces plausible-looking but WRONG numbers
# -- PR-AUC lands near 0.62 either way, because PR-AUC's floor IS the base rate. So the
# shape is checked rather than trusted.
EXPECTED = {"train_rows": 3573, "train_wins": 541, "score_rows": 1912}
actual = {"train_rows": len(train), "train_wins": int(train[LABEL].sum()),
          "score_rows": len(score)}
if actual != EXPECTED or "evidence" not in score.columns:
    raise SystemExit(
        f"STALE DATA.\n  expected {EXPECTED} + an 'evidence' column in ml_score.csv\n"
        f"  got      {actual}, evidence={'evidence' in score.columns}\n"
        "Re-export from the repo: python build_features.py && python make_dataset.py, "
        "then upload the fresh ml_train.csv / ml_score.csv."
    )
print(f"data OK for {NOTEBOOK_VERSION}")
# --------------------------------------------------------------------------------------

print(f"train  {len(train):>6,} closed deals   {train[LABEL].sum()} won "
      f"({train[LABEL].mean():.1%} win rate)")
print(f"score  {len(score):>6,} open deals     no label")
train.head()

## 2. What the features mean

| Feature | Means |
|---|---|
| `days_in_current_stage` | Days since the stage last changed |
| `days_in_sales_cycle` | Days since the deal was created |
| `stall_ratio` | `days_in_current_stage / days_in_sales_cycle` — share of the deal's life parked in one stage. **Derived, and the top driver** |
| `contract_months` | Contract term |
| `num_stakeholders` | Contacts on the deal |
| `has_champion` | A champion is named |
| `poc` | POC recorded (checkbox — see §7) |
| `opportunity_type` | new / renewal / upsell / amendment |
| `source` | outbound / inbound / events / partner / … |
| `geo` | amer / apj / emea |
| `segment` | enterprise / mid-market / smb / start-up / 2k |

Deliberately excluded: `acv` (filled after signing), `stage` (on a closed deal the
stage *is* the label), raw `created_date` (cohort censoring), `target_close_date`
(overwritten to the actual close date on 99.3% of closed deals), and 10 fields that
are only populated *because* a deal closed.

In [ ]:
print("win rate by feature value:\n")
for c in ["has_champion", "poc"]:
    g = train.groupby(c)[LABEL].agg(deals="size", wins="sum", win_rate="mean")
    lift = g.loc[1, "win_rate"] / g.loc[0, "win_rate"]
    print(f"{c}  ({lift:.1f}x lift)")
    print(g.assign(win_rate=g.win_rate.map("{:.2%}".format)).to_string(), "\n")

# opportunity_type separates harder than anything else here -- and that is the problem.
# Renewals and new business are two different sales processes; one model with one
# coefficient averages them, and 80%+ of the open pipeline is the low-rate population.
for c in ["opportunity_type", "segment", "source", "geo"]:
    g = train.groupby(c)[LABEL].agg(deals="size", wins="sum", win_rate="mean")
    g["pct_of_open"] = score[c].value_counts(normalize=True).reindex(g.index).fillna(0)
    g = g.sort_values("win_rate", ascending=False)
    spread = g.win_rate.max() / max(g.win_rate.min(), 1e-9)
    print(f"{c}  ({spread:.1f}x spread across levels)")
    print(g.assign(win_rate=g.win_rate.map("{:.2%}".format),
                   pct_of_open=g.pct_of_open.map("{:.1%}".format)).to_string(), "\n")

print("stall_ratio  (share of life parked in one stage)")
print(train.groupby(LABEL)["stall_ratio"].median().rename("median").to_string())
print("  0 = lost, 1 = won. Winners advance; losers park.")

## 2b. Look at the data before trusting any model

Six panels on the raw features. Three real problems came out of this, all already
fixed upstream in `build_features.py`:

- **`contract_months` had a 36000** — 3000 years, a units error at entry. One row, but
  it set max/p99 = 1000x and skew = 105. Anything over 120 months is now NaN'd and
  median-imputed. If panel 1 shows a long tail again, a new one has arrived.
- **`geo = "public sector"` exists only on open deals.** One-hot alignment silently
  scores those as all-zeros. `encode()` warns; panel 5 shows it as an infinite ratio.
- **`segment = "2k"` is 1.0% of train and 10.0% of open** — a 9.9x train/serve gap with
  1 win in 111 closed deals. Real segment (302 deals, 139 distinct created dates), but
  young: 185 still open, so only fast losses have resolved. Its 0.9% win rate is
  censoring, not a rate. Predictions on 2k deals are the least trustworthy in the file.

Two things that look like bugs and are **not**:

- **`stall_ratio` piles a large share of rows at exactly 1.0.** That is the deal never leaving
  its first stage, not a cap artifact. It remains the strongest single split in the
  dataset, though the margin narrowed after `DROP_LOST_WITHOUT_AMOUNT` removed the
  never-worked losses that made up most of the 1.0 bucket.
- **`num_stakeholders == 0` on 7% of train, 13% of open.** Not imputed. Blank contacts is not
  simply a bad sign -- see the panel, not this sentence, for the current rate.

Panel 6 is the one to check before tuning anything: if win rate is not monotone in a
feature's quintiles, no amount of hyperparameter search fixes it, and a tree is the
right model class precisely because it does not need monotonicity.


In [ ]:
NUM = ["days_in_current_stage", "days_in_sales_cycle", "stall_ratio",
       "contract_months", "num_stakeholders"]

fig, ax = plt.subplots(3, 2, figsize=(15, 14))

# 1. heavy tails -- log x so a units error cannot hide
for c in NUM:
    v = train[c].replace(0, np.nan).dropna()
    ax[0, 0].hist(np.log10(v + 1), bins=50, histtype="step", lw=1.4, label=c)
ax[0, 0].legend(fontsize=7)
ax[0, 0].set(xlabel="log10(value + 1)", ylabel="closed deals",
             title="1. Numeric distributions -- a spike far right is a units error")

# 2. won vs lost, standardised so five features share one axis
w, l = train[train[LABEL] == 1], train[train[LABEL] == 0]
pos = np.arange(len(NUM))
for i, c in enumerate(NUM):
    s = train[c].std() or 1.0
    ax[0, 1].barh(i - 0.2, (w[c].median() - train[c].mean()) / s, 0.4,
                  color="tab:green", label="won" if not i else "")
    ax[0, 1].barh(i + 0.2, (l[c].median() - train[c].mean()) / s, 0.4,
                  color="tab:red", label="lost" if not i else "")
ax[0, 1].axvline(0, c="k", lw=1)
ax[0, 1].set_yticks(pos); ax[0, 1].set_yticklabels(NUM, fontsize=8)
ax[0, 1].legend(fontsize=8)
ax[0, 1].set(xlabel="median, standardised (0 = overall mean)",
             title="2. Won vs lost separation -- longer opposing bars = more signal")

# 3. train vs open: same column, same meaning?
drift = pd.DataFrame({"closed": train[NUM].median(), "open": score[NUM].median()})
drift["ratio"] = (drift.open + 1e-9) / (drift.closed + 1e-9)
ax[1, 0].barh(drift.index, drift.ratio, color=["tab:orange" if abs(r - 1) > 0.25
                                               else "tab:blue" for r in drift.ratio])
ax[1, 0].axvline(1, c="k", lw=1)
ax[1, 0].set(xlabel="median(open) / median(closed)  -- 1.0 = no drift",
             title="3. Train/serve drift. days_in_sales_cycle is FINAL on closed,\n"
                   "age-so-far on open: same column, different meaning")
ax[1, 0].tick_params(labelsize=8)

# 4. feature correlation -- collinear pairs mean split importance, not more signal
corr = train[NUM].corr()
im = ax[1, 1].imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax[1, 1].set_xticks(pos); ax[1, 1].set_xticklabels(NUM, rotation=35, ha="right", fontsize=7)
ax[1, 1].set_yticks(pos); ax[1, 1].set_yticklabels(NUM, fontsize=7)
for i in range(len(NUM)):
    for j in range(len(NUM)):
        ax[1, 1].text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=7)
ax[1, 1].set_title("4. Correlation between numeric features")
plt.colorbar(im, ax=ax[1, 1], fraction=0.046)

# 5. categorical train/serve gap -- the panel that found 2k and public sector
gaps = []
for c in CATEGORICALS:
    a = train[c].value_counts(normalize=True)
    b = score[c].value_counts(normalize=True)
    for lvl in set(a.index) | set(b.index):
        ta, sa = a.get(lvl, 0.0), b.get(lvl, 0.0)
        gaps.append({"level": f"{c}={lvl}", "train": ta, "open": sa,
                     "wins": int(train.loc[train[c] == lvl, LABEL].sum()),
                     "ratio": (sa + 1e-9) / (ta + 1e-9)})
g = pd.DataFrame(gaps).sort_values("ratio")
g = pd.concat([g.head(4), g.tail(6)])
ax[2, 0].barh(g.level, np.clip(g.ratio, 0.02, 25),
              color=["tab:red" if r > 3 or r < 0.34 else "tab:blue" for r in g.ratio])
ax[2, 0].axvline(1, c="k", lw=1); ax[2, 0].set_xscale("log")
for y, (_, r) in enumerate(g.iterrows()):
    ax[2, 0].annotate(f"{r.wins} wins", (np.clip(r.ratio, 0.02, 25), y), fontsize=6.5,
                      xytext=(3, -3), textcoords="offset points")
ax[2, 0].set(xlabel="open share / train share (log, clipped to 25x)",
             title="5. Train/serve gap by level. Red = over 3x off,\nthe model barely learned it")
ax[2, 0].tick_params(labelsize=7)

# 6. monotonicity -- does win rate move consistently with the feature?
for c in NUM:
    try:
        q = pd.qcut(train[c], 5, duplicates="drop")
    except ValueError:
        continue
    rate = train.groupby(q, observed=True)[LABEL].mean()
    ax[2, 1].plot(range(len(rate)), rate.values, "o-", lw=1.4, label=c)
ax[2, 1].axhline(train[LABEL].mean(), ls="--", c="k", lw=1)
ax[2, 1].legend(fontsize=7)
ax[2, 1].set(xlabel="feature quintile, low to high", ylabel="win rate",
             title="6. Win rate by quintile -- non-monotone is fine for a tree,\n"
                   "fatal for logistic regression")
plt.tight_layout(); plt.show()

print("=== outlier / sanity checks ===")
for c in NUM:
    v = train[c]
    print(f"  {c:24} median {v.median():>8.2f}  p99 {v.quantile(.99):>8.2f}  "
          f"max {v.max():>10.2f}  max/p99 {v.max()/(v.quantile(.99)+1e-9):>7.1f}x  "
          f"skew {v.skew():>7.2f}")
worst = max(NUM, key=lambda c: train[c].max() / (train[c].quantile(.99) + 1e-9))
if train[worst].max() / (train[worst].quantile(.99) + 1e-9) > 10:
    print(f"\n  CHECK {worst}: max is >10x p99. Look for a units error at entry.")
else:
    print("\n  No feature has max >10x its p99. The 36000 contract_months is fixed.")
s1 = (train.stall_ratio == 1.0)
print(f"\nstall_ratio == 1.0 on {s1.mean():.1%} of closed deals (never left stage 1): "
      f"win rate {train.loc[s1, LABEL].mean():.1%} vs {train.loc[~s1, LABEL].mean():.1%} below it")
z = (train.num_stakeholders == 0)
print(f"num_stakeholders == 0 on {z.mean():.1%} of closed / "
      f"{(score.num_stakeholders == 0).mean():.1%} of open: win rate "
      f"{train.loc[z, LABEL].mean():.1%} vs {train[LABEL].mean():.1%} overall")


## 3. Encode

One-hot the four categoricals. The `reindex` matters: a category that appears only
in the scoring set would otherwise shift column order and silently mis-align every
feature. Using native categorical support (LightGBM/CatBoost) instead is fine —
just keep train and score consistent.

In [ ]:
def encode(df_train, df_other):
    num = [f for f in FEATURES if f not in CATEGORICALS]

    def build(df):
        out = df[num].astype(float).copy()
        for c in CATEGORICALS:
            out = pd.concat([out, pd.get_dummies(df[c], prefix=c, dtype=float)], axis=1)
        return out

    a, b = build(df_train), build(df_other)

    # The reindex below silently zeroes any level the model never trained on, so such a
    # deal scores as if the field were blank. Surface it. Also flag levels the model has
    # <=1 positive for but that carry real open-pipeline volume -- predictions there are
    # unsupported no matter what the calibration curve says.
    for c in CATEGORICALS:
        for lvl in sorted(set(df_other[c].unique()) - set(df_train[c].unique())):
            print(f"  WARNING unseen level {c}={lvl!r} on {(df_other[c]==lvl).sum()} "
                  "open deals -- scored as all-zeros, no training support")
        share = df_other[c].value_counts(normalize=True)
        w = df_train.groupby(c)[LABEL].agg(["size", "sum"])
        for lvl, r in w.iterrows():
            if r["sum"] <= 1 and share.get(lvl, 0) > 0.02:
                print(f"  WARNING thin level {c}={lvl!r}: {int(r['sum'])} win(s) in "
                      f"{int(r['size'])} closed, but {share[lvl]:.0%} of open pipeline")

    b = b.reindex(columns=a.columns, fill_value=0.0)   # align, never reorder
    return a, b

X, X_score = encode(train, score)
y = train[LABEL].to_numpy()
print(f"\n{X.shape[1]} encoded columns from {len(FEATURES)} features")

## 4. Honest evaluation first

Split by deal age — oldest 80% train, newest 20% test. **Never random.** A random
split puts a deal's future neighbours in the training set and inflates every metric.

The rows were sorted by `created_date` in §1, so the positional split below is a
time split. Do **not** drop that sort — the raw CSV order is not chronological.

In [ ]:
cut = int(len(train) * 0.8)
Xtr, Xte = X.iloc[:cut], X.iloc[cut:]
ytr, yte = y[:cut], y[cut:]
print(f"train {len(Xtr):,} / {ytr.sum()} won      test {len(Xte):,} / {yte.sum()} won")
if yte.sum() < 60:
    print(f"WARNING: {yte.sum()} test positives -- read differences of <0.02 PR-AUC as noise")

def new_model(seed=0):
    # XGBoost depth 3, min_child_weight 1. Chosen by pick_shipping.py, which runs every
    # candidate through the FULL shipping path -- same encode, same calibrator, same
    # evidence gate -- because the deliverable is a calibrated probability multiplied by
    # ACV, not a test-set PR-AUC. Measured on this split, 5 seeds:
    #
    #   model                    PR-AUC     sd    ROC   Brier   ECE    rec@20  sum(p)/act
    #   XGBoost min_child_wt1    0.6304 0.0000  0.8790  0.0668  0.0228  0.785    1.10x
    #   LightGBM (previous)      0.6206 0.0000  0.8763  0.0682  0.0272  0.747    1.11x
    #   CatBoost d3 it600 lr.03  0.6201 0.0025  0.8770  0.0677  0.0260  0.762    1.14x
    #   CatBoost d3              0.6164 0.0040  0.8766  0.0688  0.0279  0.722    1.15x
    #   CatBoost raw (uncal)     0.6159 0.0033  0.8764  0.0667  0.0260  0.724    1.15x
    #
    # XGBoost wins on all seven columns, so there is no trade-off to weigh. Both it and
    # LightGBM have seed sd 0.0000 (neither subsamples by default), so +0.0099 is not
    # seed luck. The forecast barely moves: 56.9 expected wins vs 56.2.
    #
    # Calibration is a units fix, not free accuracy. CatBoost raw ranks mid-pack yet
    # ships $0.6M more pipeline, with max probability 0.880 against XGBoost's 0.954.
    #
    # shuffle=True is load-bearing. A bare cv=3 uses UNSHUFFLED StratifiedKFold, so on
    # chronologically-sorted rows each calibration fold is a different era of the
    # business and the fitted sigmoid depends on row order -- expected wins swung 71.8
    # / 79.2 / 72.9 across three orderings of identical data, a 10% move in the
    # headline revenue number from sort order alone.
    return CalibratedClassifierCV(
        XGBClassifier(max_depth=3, n_estimators=200, learning_rate=0.05,
                      min_child_weight=1, random_state=seed, n_jobs=-1,
                      eval_metric="logloss"),
        method="sigmoid", cv=StratifiedKFold(3, shuffle=True, random_state=0))

p_test = new_model().fit(Xtr, ytr).predict_proba(Xte)[:, 1]

def recall_at_top_k(y_true, p, frac=0.2):
    k = max(1, int(len(p) * frac))
    return y_true[np.argsort(p)[::-1][:k]].sum() / y_true.sum()

print(f"""
PR-AUC        {average_precision_score(yte, p_test):.4f}   (random = {yte.mean():.4f})
Brier         {brier_score_loss(yte, p_test):.4f}   calibration -- the metric that matters
ROC-AUC       {roc_auc_score(yte, p_test):.4f}   report only, do not optimise
Recall@20%    {recall_at_top_k(yte, p_test):.4f}   share of real wins in the top-scoring fifth
sum(p)/actual {p_test.sum() / yte.sum():.2f}x    aggregate sanity, want 0.9-1.1""")
print(f"\nAccuracy is omitted on purpose: at a {yte.mean():.1%} win rate, "
      f"'everything loses'")
print(f"scores {1 - yte.mean():.1%} and finds nothing.")

In [ ]:
# Calibration: does "30%" actually close 30% of the time?
edges = np.linspace(0, 1, 11)
idx = np.clip(np.digitize(p_test, edges) - 1, 0, 9)
rel = pd.DataFrame([
    {"bin": f"{edges[b]:.0%}-{edges[b+1]:.0%}", "deals": int((idx == b).sum()),
     "predicted": p_test[idx == b].mean(), "actual": yte[idx == b].mean()}
    for b in range(10) if (idx == b).sum()
])
display(rel.round(4))

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot([0, 1], [0, 1], "k--", lw=1, label="perfect")
ax[0].plot(rel.predicted, rel.actual, "o-", label="model")
for _, r in rel.iterrows():
    ax[0].annotate(f"n={r.deals:.0f}", (r.predicted, r.actual), fontsize=7,
                   xytext=(4, -9), textcoords="offset points")
ax[0].set(xlabel="predicted", ylabel="actual win rate", title="Reliability")
ax[0].legend()

imp = permutation_importance(new_model().fit(Xtr, ytr), Xte, yte,
                             scoring="average_precision", n_repeats=10, random_state=0)
top = pd.Series(imp.importances_mean, index=Xte.columns).nlargest(10).sort_values()
ax[1].barh(top.index, top.values)
ax[1].set(title="Permutation importance (PR-AUC drop when shuffled)")
plt.tight_layout(); plt.show()

thin = rel[rel.deals < 20]
big = rel.loc[rel.deals.idxmax()]
print(f"{len(thin)} of {len(rel)} bins hold under 20 deals ({thin.deals.min():.0f}-"
      f"{thin.deals.max():.0f}) -- that wobble is small-sample noise, not miscalibration.")
print(f"The {big['bin']} bin holds {big.deals:.0f} deals and is the one to trust: "
      f"predicted {big.predicted:.4f} vs actual {big.actual:.4f}.")

## 4b. Is it overfitting? Separate the two gaps first

A raw train-vs-test PR-AUC gap on this dataset reads as catastrophic overfitting and
mostly is not. **PR-AUC is not comparable across sets with different base rates** --
its floor *is* the base rate. Train is 16.2% positives, test 11.0%, a 1.46x ratio, so
PR-AUC drops mechanically on an identical model.

Three splits pull the two effects apart:

| split | rows | base rate | what a drop here means |
|---|---|---|---|
| train, in-sample | 2,858 | 16.2% | — |
| train, out-of-fold | 2,858 | 16.2% | unseen rows, same base rate → **overfitting** |
| test, time split | 715 | 11.0% | unseen rows, different era → **distribution shift** |

The cell also reports **lift** (PR-AUC ÷ base rate), which is comparable across sets,
and **ROC-AUC**, which is base-rate invariant.

Measured: in-sample 0.917 → OOF 0.860 is a 0.057 overfitting gap. OOF → test 0.860 →
0.630 is 0.230, four times larger, and both sets are unseen. Normalised, the model
discriminates *better* on test: 5.3x lift out-of-fold vs 5.7x on test.

The shift is cohort censoring, not memorisation. Test deals are the newest, so the
youngest; among young deals only the fast losses have resolved while their eventual
wins are still open and sitting in `ml_score.csv`. That is the whole 16.2% → 11.0% move.

The regularisation sweep below is the direct check: if this were overfitting, more
regularisation would help the test number. It does not -- `min_child_samples` 50 → 400
costs 0.081 PR-AUC, and the least-regularised config (depth8/mcs5, in-sample 1.000)
is the *worst* on test at 0.545 while `depth2 num_leaves4` ties shipping at 0.619. On the pre-filter dataset the widest-gap config also won on test, which said the gap
was measuring nothing. That no longer holds: depth8/mcs5 memorises train perfectly
(1.000) and is last on test, so *some* of the gap is now real overfitting. The 0.057
in-sample→OOF gap still bounds it, and it is four times smaller than the 0.230 shift,
so the shift is still the dominant term -- but the sweep is no longer evidence that
overfitting is absent, only that regularising past the current setting does not help.


In [ ]:
from sklearn.model_selection import cross_val_predict

p_in = new_model().fit(Xtr, ytr).predict_proba(Xtr)[:, 1]
# Out-of-fold within train: unseen rows at the SAME base rate. This is the only
# comparison that isolates overfitting from distribution shift.
p_oof = cross_val_predict(new_model(), Xtr, ytr, method="predict_proba",
                          cv=StratifiedKFold(5, shuffle=True, random_state=0))[:, 1]

def gap_row(name, yy, pp):
    ap = average_precision_score(yy, pp)
    return {"split": name, "n": len(yy), "wins": int(yy.sum()), "base_rate": yy.mean(),
            "PR_AUC": ap, "lift_over_random": ap / yy.mean(),
            "ROC": roc_auc_score(yy, pp), "Brier": brier_score_loss(yy, pp)}

gaps = pd.DataFrame([gap_row("train in-sample", ytr, p_in),
                     gap_row("train OUT-of-fold", ytr, p_oof),
                     gap_row("test (time split)", yte, p_test)])
display(gaps.round({"base_rate": 4, "PR_AUC": 4, "lift_over_random": 1,
                    "ROC": 4, "Brier": 5}))

ap_in, ap_oof, ap_te = gaps.PR_AUC
print(f"in-sample -> OOF  (same base rate) : {ap_in:.4f} -> {ap_oof:.4f}   "
      f"gap {ap_in - ap_oof:+.4f}  <- OVERFITTING")
print(f"OOF -> test  (base rate {y.mean():.1%}->{yte.mean():.1%}): {ap_oof:.4f} -> {ap_te:.4f}   "
      f"gap {ap_oof - ap_te:+.4f}  <- DISTRIBUTION SHIFT")
print(f"\nlift over random: train {gaps.lift_over_random[1]:.1f}x, test "
      f"{gaps.lift_over_random[2]:.1f}x  <- higher on test, once the base rate is out")
print(f"ROC (base-rate invariant): {gaps.ROC[0]:.4f} in-sample, {gaps.ROC[1]:.4f} OOF, "
      f"{gaps.ROC[2]:.4f} test")

# Direct test: if this is overfitting, MORE regularisation helps the test number.
print("\n=== does regularisation help TEST? (3 seeds) ===")
REG = {"SHIPPING d3/leaf8/mcs50": {},
       "depth2 num_leaves4":      dict(max_depth=2, num_leaves=4),
       "min_child_samples 200":   dict(min_child_samples=200),
       "min_child_samples 400":   dict(min_child_samples=400),
       "n_estimators 50":         dict(n_estimators=50),
       "L1/L2 + subsample":       dict(reg_alpha=5.0, reg_lambda=5.0, subsample=0.7,
                                       subsample_freq=1, colsample_bytree=0.7),
       "depth8 mcs5 (LESS reg)":  dict(max_depth=8, num_leaves=64, min_child_samples=5)}
for nm, kw in REG.items():
    te, ins = [], []
    for s in range(3):
        params = dict(max_depth=3, num_leaves=8, min_child_samples=50,
                      n_estimators=200, learning_rate=0.05, random_state=s,
                      verbose=-1, n_jobs=-1)
        params.update(kw)          # kw overrides, rather than colliding with, the base
        m = CalibratedClassifierCV(
            LGBMClassifier(**params), method="sigmoid",
            cv=StratifiedKFold(3, shuffle=True, random_state=0)).fit(Xtr, ytr)
        te.append(average_precision_score(yte, m.predict_proba(Xte)[:, 1]))
        ins.append(average_precision_score(ytr, m.predict_proba(Xtr)[:, 1]))
    print(f"  {nm:26} test {np.mean(te):.4f}   in-sample {np.mean(ins):.4f}   "
          f"gap {np.mean(ins) - np.mean(te):+.4f}")
print("\nRead the LAST row against the FIRST. If a wider gap meant worse")
print("generalisation, the least-regularised model would be worst on test. It is not.")


## 4c. Why this model — 26 configurations compared

**XGBoost is the shipping model as of this section.** It was chosen here, on measured
results, and the choice is recorded in `new_estimator()` in `make_dataset.py`.

Every row is a tree model except the logistic regression, which is in as a sanity
floor. Five families:

| family | how it grows | configs |
|---|---|---|
| one tree | a single greedy tree | 2 |
| bagging | average many independent trees | 4 |
| HistGB / AdaBoost | sklearn boosting, level-wise | 8 |
| LightGBM / XGBoost | leaf-wise boosting | 5 |
| CatBoost | **symmetric (oblivious)** trees — every node at a depth splits on the same feature | 6 |

CatBoost is worth having in for a structural reason, not just for another number: its
symmetric trees are a genuinely different inductive bias. That is stronger implicit
regularisation than leaf-wise growth, so it holds up at 492 positives without any
`min_data_in_leaf` tuning — and if it landed far from the leaf-wise boosters, that would say the
result depends on how trees are grown rather than on the data. It does not.

The cell installs `lightgbm`, `xgboost` and `catboost` itself on Colab. Locally,
`pip install lightgbm xgboost catboost` — on a Mac the first two also need the OpenMP
runtime (`brew install libomp`). Without them the cell drops those rows and still runs.

Expect ~2–3 minutes: 26 configs × 5 seeds, and the calibrator fits each model 3 more
times for its CV folds.

Four things this table is here to settle:
- **Ensembling isn't decoration.** A single tree is the *worst* tree in the set. At 49
  test wins one tree either underfits or memorises a handful of deals.
- **Calibration is not free accuracy.** `scale_pos_weight` / `auto_class_weights` and
  deeper trees can win on PR-AUC while making `sum(p)/actual` worse — and that column
  is the revenue number. Rank on `calib_err` and Brier, not PR-AUC alone.
- **A gap inside the seed spread is not a result.** `pr_sd` is printed per row for
  exactly this reason. XGBoost measured 0.6304 ±0.0000 against LightGBM's 0.6206
  ±0.0000 — both deterministic, so that +0.0099 is a real gap rather than a lucky
  seed, and XGBoost also wins Brier, ECE, ROC, recall@20% and `sum(p)/actual`. Seven
  columns, no trade-off. Contrast the CatBoost family, which sits inside its own
  ±0.0040 spread of LightGBM and is genuinely tied.
- **The three real boosters land within 0.02 of each other.** LightGBM 0.621, CatBoost
  0.629, XGBoost 0.615 — while bagging sits at 0.51 and a single tree at 0.42. The
  gradient-boosting *family* is what matters here; which implementation is close to
  arbitrary, and any of the three would ship.


In [ ]:
import time

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier,
                              AdaBoostClassifier, BaggingClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

if IN_COLAB:
    !pip install -q lightgbm xgboost catboost
try:
    from lightgbm import LGBMClassifier
    from xgboost import XGBClassifier
    HAVE_BOOSTERS = True
except ImportError:
    HAVE_BOOSTERS = False
    print("lightgbm/xgboost not installed -- skipping 5 rows.")
    print("  pip install lightgbm xgboost   (on a Mac also: brew install libomp)")
try:
    from catboost import CatBoostClassifier
    HAVE_CATBOOST = True
except ImportError:
    HAVE_CATBOOST = False
    print("catboost not installed -- skipping 6 rows.  pip install catboost")

SEEDS = range(5)
skf = lambda: StratifiedKFold(3, shuffle=True, random_state=0)
cal = lambda m: CalibratedClassifierCV(m, method="sigmoid", cv=skf())
pos_weight = (ytr == 0).sum() / (ytr == 1).sum()   # ~5:1 at a 16.2% win rate

def hgb(s, **kw):
    p = dict(max_depth=3, min_samples_leaf=50, max_iter=200, learning_rate=0.05,
             early_stopping=True, validation_fraction=0.15, random_state=s)
    p.update(kw)
    return HistGradientBoostingClassifier(**p)

FAMILIES = {
    # --- not a tree: the floor any tree has to beat --------------------------
    "logreg L2 (not a tree)":  lambda s: cal(make_pipeline(StandardScaler(),
                                   LogisticRegression(max_iter=2000, C=0.1))),
    # --- one tree, no ensemble ----------------------------------------------
    "single tree depth3":      lambda s: cal(DecisionTreeClassifier(max_depth=3,
                                   min_samples_leaf=50, random_state=s)),
    "single tree unlimited":   lambda s: cal(DecisionTreeClassifier(random_state=s)),
    # --- bagging: average many independent trees -----------------------------
    "random forest 500":       lambda s: cal(RandomForestClassifier(n_estimators=500,
                                   min_samples_leaf=50, random_state=s, n_jobs=-1)),
    "random forest balanced":  lambda s: cal(RandomForestClassifier(n_estimators=500,
                                   min_samples_leaf=50, class_weight="balanced",
                                   random_state=s, n_jobs=-1)),
    "extra trees 500":         lambda s: cal(ExtraTreesClassifier(n_estimators=500,
                                   min_samples_leaf=50, random_state=s, n_jobs=-1)),
    "bagged tree depth3":      lambda s: cal(BaggingClassifier(
                                   DecisionTreeClassifier(max_depth=3, min_samples_leaf=50),
                                   n_estimators=200, random_state=s, n_jobs=-1)),
    # --- boosting: each tree fixes the previous ones' mistakes ---------------
    "adaboost 200":            lambda s: cal(AdaBoostClassifier(n_estimators=200,
                                   learning_rate=0.05, random_state=s)),
    "HGB raw (uncalibrated)":  lambda s: hgb(s),
    "HGB+sigmoid (was shipping)": lambda s: cal(hgb(s)),
    "HGB+isotonic":            lambda s: CalibratedClassifierCV(hgb(s),
                                   method="isotonic", cv=skf()),
    "HGB depth2 shallower":    lambda s: cal(hgb(s, max_depth=2)),
    "HGB depth8 leaf5":        lambda s: cal(hgb(s, max_depth=8, min_samples_leaf=5)),
    "HGB lr0.02 iter800":      lambda s: cal(hgb(s, learning_rate=0.02, max_iter=800)),
    "HGB no early stopping":   lambda s: cal(hgb(s, early_stopping=False)),
}
if HAVE_BOOSTERS:
    FAMILIES.update({
        "LightGBM (was shipping)": lambda s: cal(LGBMClassifier(max_depth=3, num_leaves=8,
                                      min_child_samples=50, n_estimators=200, learning_rate=0.05,
                                      random_state=s, verbose=-1, n_jobs=-1)),
        "LightGBM scale_pos_wt":  lambda s: cal(LGBMClassifier(max_depth=3, num_leaves=8,
                                      min_child_samples=50, n_estimators=200, learning_rate=0.05,
                                      scale_pos_weight=pos_weight,
                                      random_state=s, verbose=-1, n_jobs=-1)),
        "LightGBM subsample+reg": lambda s: cal(LGBMClassifier(max_depth=3, num_leaves=8,
                                      min_child_samples=50, n_estimators=200, learning_rate=0.05,
                                      reg_alpha=1.0, reg_lambda=1.0, subsample=0.8,
                                      subsample_freq=1, colsample_bytree=0.8,
                                      random_state=s, verbose=-1, n_jobs=-1)),
        "XGBoost":                lambda s: cal(XGBClassifier(max_depth=3, min_child_weight=50,
                                      n_estimators=200, learning_rate=0.05, random_state=s,
                                      eval_metric="logloss", n_jobs=-1)),
        "XGBoost  SHIPPING":      lambda s: cal(XGBClassifier(max_depth=3, min_child_weight=1,
                                      n_estimators=200, learning_rate=0.05, random_state=s,
                                      eval_metric="logloss", n_jobs=-1)),
    })
if HAVE_CATBOOST:
    # CatBoost grows SYMMETRIC (oblivious) trees -- every node at a given depth splits on
    # the same feature. That is stronger implicit regularisation than LightGBM's leaf-wise
    # growth, which is why it stays sane at 492 positives without min_child_samples
    # tuning. allow_writing_files=False stops it littering catboost_info/ into the cwd.
    def cbc(s, **kw):
        p = dict(depth=3, iterations=200, learning_rate=0.05, min_data_in_leaf=50,
                 random_seed=s, verbose=0, allow_writing_files=False, thread_count=-1)
        p.update(kw)
        return CatBoostClassifier(**p)

    FAMILIES.update({
        "CatBoost d3":             lambda s: cal(cbc(s)),
        "CatBoost d6 (its dflt)":  lambda s: cal(cbc(s, depth=6, min_data_in_leaf=1)),
        "CatBoost d3 it600 lr.03": lambda s: cal(cbc(s, iterations=600, learning_rate=0.03)),
        "CatBoost raw (uncal)":    lambda s: cbc(s),
        "CatBoost auto_class_wt":  lambda s: cal(cbc(s, auto_class_weights="Balanced")),
        "CatBoost l2_leaf_reg 10": lambda s: cal(cbc(s, l2_leaf_reg=10.0)),
    })

SHIP_NAME = "XGBoost  SHIPPING" if HAVE_BOOSTERS else "HGB+sigmoid (was shipping)"

rows = []
PREDS = {}     # model -> seed-0 test predictions, for the plots in 4d/4e
PER_SEED = []  # one row per (model, seed), for the per-model detail in 4e
for name, make in FAMILIES.items():
    pr, br, rc, ratio, secs = [], [], [], [], []
    for s in SEEDS:
        t0 = time.time()
        p = make(s).fit(Xtr, ytr).predict_proba(Xte)[:, 1]
        secs.append(time.time() - t0)
        if s == 0:
            PREDS[name] = p
        pr.append(average_precision_score(yte, p))
        br.append(brier_score_loss(yte, p))
        rc.append(roc_auc_score(yte, p))
        ratio.append(p.sum() / yte.sum())
        PER_SEED.append({"model": name, "seed": s, "PR_AUC": pr[-1], "Brier": br[-1],
                         "ROC": rc[-1], "sum_p_over_actual": ratio[-1],
                         "recall_at_20pct": recall_at_top_k(yte, p),
                         "p_max": p.max(), "p_median": np.median(p), "fit_s": secs[-1]})
    rows.append({"model": name, "PR_AUC": np.mean(pr), "pr_sd": np.std(pr),
                 "Brier": np.mean(br), "ROC": np.mean(rc),
                 "sum_p_over_actual": np.mean(ratio), "fit_s": np.mean(secs)})
    print(f"  done {name}")

per_seed = pd.DataFrame(PER_SEED)

cmp = pd.DataFrame(rows).sort_values("PR_AUC", ascending=False).reset_index(drop=True)
# calib_err is the product metric: how far the revenue roll-up would be off.
cmp["calib_err"] = (cmp.sum_p_over_actual - 1).abs()
print(f"\ntrain {len(Xtr):,}/{ytr.sum()} wins   test {len(Xte):,}/{yte.sum()} wins   "
      f"{len(list(SEEDS))} seeds each   random PR-AUC = {yte.mean():.4f}\n")
display(cmp.round({"PR_AUC": 4, "pr_sd": 4, "Brier": 5, "ROC": 4,
                   "sum_p_over_actual": 2, "fit_s": 2, "calib_err": 2}))

ship = cmp.loc[cmp.model == SHIP_NAME].iloc[0]
best_pr, best_cal = cmp.iloc[0], cmp.sort_values("calib_err").iloc[0]
print(f"best PR-AUC      {best_pr.model:24}{best_pr.PR_AUC:.4f}  "
      f"(shipping {ship.PR_AUC:.4f}, gap {best_pr.PR_AUC - ship.PR_AUC:+.4f})")
print(f"best calibrated  {best_cal.model:24}sum(p)/actual {best_cal.sum_p_over_actual:.2f}x")
print(f"\nSeed spread on the shipping model is +-{ship.pr_sd:.4f}. Treat any PR-AUC gap")
print("smaller than that as noise -- it is not a reason to switch models.")


## 4d. The same 26 models, as pictures

The table ranks. These plots show *why* a model ranks where it does — and they
surface failures a single number hides.

Six panels:

1. **PR curve** — precision against recall. The only curve that tells you what a
   rep experiences: "of the deals this flagged, how many closed?" Dashed line is
   the 11.0% test base rate.
2. **Reliability** — predicted vs actual, decile bins. On the diagonal = "40%
   means 40%". Below it = over-promising. This is the panel that catches a model
   the PR-AUC column calls excellent.
3. **PR-AUC vs calibration error** — the tradeoff, as a scatter. Bottom-right is
   what you want: ranks well *and* the revenue roll-up is right. A model can be
   far right and far up, which means it sorts deals correctly while getting the
   forecast total wrong.
4. **Seed spread** — mean PR-AUC with the ±1 s.d. bar across 5 seeds. Bars that
   overlap are the same model as far as this test set can tell.
5. **Score distributions** — where each model puts its probability mass. A model
   that piles everything at 0.02 has good Brier and is useless.
6. **Cumulative wins captured** — walk the ranked list; how fast do you collect
   the 49 real wins? Diagonal is random. This is the lift a rep actually gets from
   working the list top-down.

Panels 1, 2, 5 and 6 use **seed 0 only**, since a curve can't be averaged
honestly. Panels 3 and 4 use all 5 seeds. Run 4c first — this cell reads `PREDS`
and `cmp` from it.


In [ ]:
if "PREDS" not in globals():
    raise NameError("run section 4c first -- this cell reads PREDS and cmp from it")

order = cmp.model.tolist()                      # best PR-AUC first
colors = plt.cm.viridis(np.linspace(0, 0.92, len(order)))
CLR = dict(zip(order, colors))
SHIP = SHIP_NAME
lw = lambda m: 2.6 if m == SHIP else 1.2        # shipping model drawn thicker
from sklearn.metrics import precision_recall_curve

fig, ax = plt.subplots(3, 2, figsize=(15, 15))

# 1. precision-recall -- what a rep working the flagged list actually gets
for m in order:
    pre, rec, _ = precision_recall_curve(yte, PREDS[m])
    ax[0, 0].plot(rec, pre, color=CLR[m], lw=lw(m), alpha=0.9)
ax[0, 0].axhline(yte.mean(), ls="--", c="k", lw=1)
ax[0, 0].set(xlabel="recall (share of real wins found)", ylabel="precision",
             title="1. Precision-Recall (seed 0)", ylim=(0, 1))
ax[0, 0].annotate(f"base rate {yte.mean():.1%}", (0.55, yte.mean()),
                  xytext=(0, 6), textcoords="offset points", fontsize=8)

# 2. reliability -- does "40%" close 40% of the time?
edges = np.linspace(0, 1, 11)
for m in order:
    p = PREDS[m]
    b = np.clip(np.digitize(p, edges) - 1, 0, 9)
    pts = [(p[b == k].mean(), yte[b == k].mean()) for k in range(10)
           if (b == k).sum() >= 20]     # <20 deals in a bin is noise, not a point
    if pts:
        ax[0, 1].plot(*zip(*pts), "o-", color=CLR[m], lw=lw(m), ms=4, alpha=0.9)
ax[0, 1].plot([0, 1], [0, 1], "k--", lw=1)
ax[0, 1].set(xlabel="predicted", ylabel="actual win rate",
             title="2. Reliability -- on the diagonal or over-promising (bins >=20 deals)")

# 3. the real tradeoff: ranking quality vs revenue accuracy
for m in order:
    r = cmp.loc[cmp.model == m].iloc[0]
    ax[1, 0].scatter(r.PR_AUC, r.calib_err, color=CLR[m],
                     s=190 if m == SHIP else 70,
                     marker="*" if m == SHIP else "o", zorder=3)
    ax[1, 0].annotate(m, (r.PR_AUC, r.calib_err), fontsize=6.5,
                      xytext=(4, 3), textcoords="offset points")
ax[1, 0].axhline(0, c="k", lw=1)
ax[1, 0].set(xlabel="PR-AUC (higher better) ->", ylabel="|sum(p)/actual - 1|  (lower better)",
             title="3. Ranking vs revenue accuracy -- want bottom-right (* = shipping)")

# 4. seed spread -- overlapping bars mean the gap is not real
c4 = cmp.sort_values("PR_AUC")
ax[1, 1].barh(c4.model, c4.PR_AUC, xerr=c4.pr_sd,
              color=[CLR[m] for m in c4.model], error_kw=dict(lw=1.1, ecolor="k"))
ax[1, 1].axvline(yte.mean(), ls="--", c="r", lw=1)
ax[1, 1].set(xlabel="PR-AUC (bar = mean of 5 seeds, whisker = +-1 s.d.)",
             title="4. Overlapping whiskers = same model, as far as 49 wins can tell")
ax[1, 1].tick_params(labelsize=7)

# 5. where each model puts its probability mass
for m in order:
    ax[2, 0].hist(PREDS[m], bins=np.linspace(0, 1, 41), histtype="step",
                  color=CLR[m], lw=lw(m), alpha=0.9)
ax[2, 0].set(xlabel="predicted win probability", ylabel="test deals (log)",
             yscale="log", title="5. Score distributions -- all mass at zero = useless")

# 6. cumulative wins captured walking the ranked list top-down
n = len(yte)
for m in order:
    hits = np.cumsum(yte[np.argsort(PREDS[m])[::-1]]) / yte.sum()
    ax[2, 1].plot(np.arange(1, n + 1) / n, hits, color=CLR[m], lw=lw(m), alpha=0.9)
ax[2, 1].plot([0, 1], [0, 1], "k--", lw=1)
ax[2, 1].axvline(0.2, ls=":", c="r", lw=1)
ax[2, 1].set(xlabel="share of pipeline worked, best-scored first",
             ylabel=f"share of the {int(yte.sum())} real wins captured",
             title="6. Lift -- dashed is random, red line is the top 20%")

h = [plt.Line2D([], [], color=CLR[m], lw=lw(m)) for m in order]
fig.legend(h, order, loc="lower center", ncol=4, fontsize=7.5,
           frameon=False, bbox_to_anchor=(0.5, -0.035))
plt.tight_layout(); plt.show()

top = cmp.nsmallest(1, "calib_err").iloc[0]
print(f"Panel 3 read-off: best-calibrated is {top.model} at {top.calib_err:.2f} off, "
      f"PR-AUC {top.PR_AUC:.4f}")
print(f"                  best-ranking is {cmp.iloc[0].model} at PR-AUC "
      f"{cmp.iloc[0].PR_AUC:.4f}, but {cmp.iloc[0].calib_err:.2f} off on revenue")
print("\nPanel 2 is the veto. A model that ranks well but sits below the diagonal")
print("hands sales numbers that do not hold up, which is worse than a weaker model")
print("that is honest about its own uncertainty.")


## 4e. Every model, one at a time

Section 4d overlays all 26 series in six axes, which is fine for spotting the
outliers and useless for reading any single model. This section breaks them out.

Three things per model:

- **A per-seed table.** Five rows, one per seed, so you can see the variation
  directly instead of inferring it from a `±` column. If a model's PR-AUC swings
  0.38 → 0.52 across seeds, its mean is not a description of anything.
- **Its own reliability panel** in a small-multiples grid, with the deal count
  written on each bin. On the diagonal means calibrated. Bins with under 20
  deals are drawn hollow — at 79 test wins those points move on a single deal.
- **A verdict line** naming the specific failure mode, if it has one:
  over-promising, mass collapsed at zero, or seed spread wider than its own
  advantage over the shipping model.

`MODEL_REPORT` at the bottom is a plain DataFrame — sort or filter it however
you want. Run 4c first.


In [ ]:
if "per_seed" not in globals():
    raise NameError("run section 4c first -- this cell reads per_seed, PREDS and cmp")

# ---- per-seed detail, every model, every seed --------------------------------
pd.set_option("display.max_rows", 200)
detail = per_seed.copy()
detail["model"] = pd.Categorical(detail.model, categories=cmp.model.tolist(), ordered=True)
detail = detail.sort_values(["model", "seed"]).reset_index(drop=True)
print(f"{len(cmp)} models x {detail.seed.nunique()} seeds, "
      f"tested on {len(yte):,} deals / {int(yte.sum())} wins\n")
display(detail.round({"PR_AUC": 4, "Brier": 5, "ROC": 4, "sum_p_over_actual": 2,
                      "recall_at_20pct": 3, "p_max": 3, "p_median": 4, "fit_s": 2}))

# ---- one reliability panel per model ----------------------------------------
edges = np.linspace(0, 1, 11)
order = cmp.model.tolist()
ncol = 4
nrow = int(np.ceil(len(order) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(4 * ncol, 3.4 * nrow),
                         sharex=True, sharey=True)
axes = np.atleast_1d(axes).ravel()

report = []
ship_pr = cmp.loc[cmp.model == SHIP_NAME].iloc[0].PR_AUC

for ax, m in zip(axes, order):
    p = PREDS[m]
    r = cmp.loc[cmp.model == m].iloc[0]
    b = np.clip(np.digitize(p, edges) - 1, 0, 9)

    ax.plot([0, 1], [0, 1], "k--", lw=1)
    solid_x, solid_y = [], []
    for k in range(10):
        n = int((b == k).sum())
        if not n:
            continue
        px, ay = p[b == k].mean(), yte[b == k].mean()
        thin = n < 20          # hollow = too few deals to trust the point
        ax.plot(px, ay, "o", ms=6, mfc="none" if thin else "tab:blue",
                mec="tab:blue", mew=1.3)
        ax.annotate(f"{n}", (px, ay), fontsize=6, xytext=(4, -8),
                    textcoords="offset points", color="0.35")
        if not thin:
            solid_x.append(px); solid_y.append(ay)
    if len(solid_x) > 1:
        ax.plot(solid_x, solid_y, "-", color="tab:blue", lw=1.6)

    # signed calibration bias: >1 over-promises, <1 under-promises
    over = r.sum_p_over_actual
    ax.set_title(f"{m}\nPR-AUC {r.PR_AUC:.3f}+-{r.pr_sd:.3f}   "
                 f"sum(p)/act {over:.2f}x", fontsize=8.5)
    ax.set_xlim(-0.03, 1.03); ax.set_ylim(-0.03, 1.03)

    flags = []
    if over > 1.35:
        flags.append(f"over-promises {over:.2f}x")
    elif over < 0.75:
        flags.append(f"under-promises {over:.2f}x")
    if p.max() < 0.30:
        flags.append(f"mass collapsed, max p only {p.max():.2f}")
    if r.pr_sd > abs(r.PR_AUC - ship_pr) and m != SHIP_NAME:
        flags.append("seed spread exceeds its gap vs shipping")
    if r.PR_AUC < yte.mean() * 3:
        flags.append("barely above base rate")
    report.append({"model": m, "PR_AUC": r.PR_AUC, "pr_sd": r.pr_sd,
                   "Brier": r.Brier, "sum_p_over_actual": over,
                   "verdict": "; ".join(flags) if flags else "no flag"})

for i, ax in enumerate(axes[:len(order)]):
    if i % ncol == 0:
        ax.set_ylabel("actual win rate", fontsize=8)
    if i >= len(order) - ncol:
        ax.set_xlabel("predicted", fontsize=8)
for ax in axes[len(order):]:
    ax.axis("off")
fig.suptitle("Reliability, one panel per model (hollow = bin under 20 deals, "
             "number = deals in bin)", y=1.002)
plt.tight_layout(); plt.show()

MODEL_REPORT = pd.DataFrame(report)
print("\nper-model verdicts, best PR-AUC first:\n")
display(MODEL_REPORT.round({"PR_AUC": 4, "pr_sd": 4, "Brier": 5,
                            "sum_p_over_actual": 2}))
clean = MODEL_REPORT[MODEL_REPORT.verdict == "no flag"]
print(f"{len(clean)} of {len(MODEL_REPORT)} models carry no flag: "
      f"{', '.join(clean.model) if len(clean) else 'none'}")
print("\nA flag is not a disqualification -- it names what to check before")
print("shipping that model. 'seed spread exceeds its gap vs shipping' means the")
print("model may simply be the shipping model with a luckier seed.")


## 5. Train on everything, score the open pipeline

Evaluation is done. Refit on all 3,573 closed deals — more labels, better model —
and score the open ones.

In [ ]:
# Shuffle before the final fit. The calibrator's folds are positional slices of
# whatever order arrives, so on the created_date-sorted frame each fold is a different
# era of the business. Fixed seed keeps the run reproducible.
shuf = train.sample(frac=1, random_state=0).reset_index(drop=True)
Xs, X_score = encode(shuf, score)
final = new_model().fit(Xs, shuf[LABEL].to_numpy())

# THE EVIDENCE GATE. Mirrors make_dataset.py -- keep the two in step, or this notebook
# publishes one forecast while the pipeline publishes another.
#
# `evidence` counts how many independent things a rep actually recorded: champion, POC,
# >=2 contacts, advanced at least one stage, real ACV. Below 2 the deal has nothing on
# it, and the matching closed cohort wins 1.2% of the time against 20.0% at evidence=2.
# The model reads absence of data on those rows, so its output is not a forecast.
#
# It gates on EVIDENCE, not stage: a 1-profile deal with champion, POC and stakeholders
# filled in clears the gate and keeps its score, which is exactly the signal worth
# having. Only absence is censored.
#
# win_probability is NULLED there rather than left low, because a number in that cell
# gets summed by whoever opens the CSV: totalling all 1,912 gave $5.67M against a
# committed $4.1M. raw_probability keeps the model's output for auditing, under a name
# nobody totals by accident.
MIN_EVIDENCE = 2

out = score[["display_id", "acv", "evidence", "stage_name"] + FEATURES].copy()
too_early = out.evidence < MIN_EVIDENCE
out["raw_probability"] = final.predict_proba(X_score)[:, 1]
out["win_probability"] = out.raw_probability.mask(too_early)
out["health"] = pd.cut(out.win_probability, [-0.01, 0.10, 0.30, 1.0],
                       labels=["Red", "Yellow", "Green"])
out["health"] = out.health.cat.add_categories(["Too early"])
out.loc[too_early, "health"] = "Too early"
out["expected_revenue"] = out.win_probability * out.acv
out = out.sort_values("win_probability", ascending=False, na_position="last")

too_early = out.evidence < MIN_EVIDENCE          # realign after the sort
scored, early = out[~too_early], out[too_early]

assert out.raw_probability.between(0, 1).all()
assert out.health.notna().all()
assert out.win_probability.isna().equals(too_early), "NULLs do not match the gate"
assert out.loc[too_early, "expected_revenue"].isna().all(), "revenue on a gated deal"

print(f"{len(out):,} open deals\n")
print(f"COMMITTED (evidence >= {MIN_EVIDENCE})  {len(scored):,} deals  "
      f"{scored.win_probability.sum():.0f} expected wins  "
      f"${scored.expected_revenue.sum()/1e6:.1f}M")
print(f"  win_probability   min {scored.win_probability.min():.3f}  "
      f"median {scored.win_probability.median():.3f}  "
      f"max {scored.win_probability.max():.3f}")
print("    ^ amount is real on ~41% of these deals; the rest is imputed. Range, not point.")
print(f"TOO EARLY                 {len(early):,} deals  win_probability NULL")
print(f"  (would have added {early.raw_probability.sum():.0f} wins / "
      f"${(early.raw_probability * early.acv).sum()/1e6:.1f}M -- excluded on purpose)")
print("\nhealth bands (thresholds are PLACEHOLDERS -- sales owns these numbers):")
print(out.health.value_counts().reindex(["Green", "Yellow", "Red", "Too early"]).to_string())

out.to_csv("ml_predictions.csv", index=False)
print("\nwrote ml_predictions.csv")
out[["display_id", "stage_name", "evidence", "win_probability", "health", "acv",
     "stall_ratio", "has_champion", "segment"]].head(10).round(3)

In [ ]:
if IN_COLAB:
    from google.colab import files
    files.download("ml_predictions.csv")

## 6. The output a sales manager would act on

In [ ]:
risk = out[(out.acv > 50_000) & (out.win_probability < 0.10)] \
         .nlargest(15, "days_in_current_stage")
print(f"AT RISK -- >$50k, <10% win probability, longest stalled")
print(f"${risk.acv.sum()/1e6:.1f}M across these {len(risk)} deals, "
      "all booked at full stage-constant value by today's forecast\n")
display(risk[["display_id", "acv", "win_probability", "days_in_current_stage",
              "stall_ratio", "num_stakeholders", "has_champion", "segment",
              "opportunity_type"]].round(3).reset_index(drop=True))

print("CHECK before presenting: if these share a created-date cluster and all have")
print("0 stakeholders, they are a bulk import that was never worked -- a data-hygiene")
print("story, not a rep-behaviour story. Different fix, different audience.")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].hist(out.win_probability, bins=40, color="tab:blue")
ax[0].set(yscale="log", xlabel="win probability", ylabel="deals (log)",
          title="Open pipeline scores")

band = out.groupby("health", observed=True).agg(
    deals=("display_id", "size"), revenue=("expected_revenue", "sum"))
ax[1].bar(band.index.astype(str), band.revenue / 1e6,
          color=["tab:red", "tab:orange", "tab:green"])
ax[1].set(ylabel="expected revenue ($M)", title="Weighted pipeline by health")
for i, (d, r) in enumerate(zip(band.deals, band.revenue)):
    ax[1].annotate(f"{d} deals", (i, r/1e6), ha="center", va="bottom", fontsize=9)
plt.tight_layout(); plt.show()

## 7. Limits — read before quoting any number above

**Metrics are optimistic.** Training rows are closed deals in their *final* state;
scoring rows are open deals *mid-flight*. `days_in_sales_cycle` means "final cycle
length" on a closed deal but "age so far" on an open one — same column, two
meanings. Sizing that gap needs backtesting, which needs daily snapshots that
don't exist yet.

**Calibration always extrapolates forward.** 89% of the deals being scored were
created in 2026, while most training labels are 2024–2025 — a resolved cohort can't
be recent, by definition. Narrowing the training window doesn't fix it (measured:
it degrades Σp/actual from 1.36× to 1.54×). This is why Brier and the reliability
curve matter more here than PR-AUC.

**Revenue is softer than the probability.** Amount is real on only ~41% of open
deals. The rest is imputed as a **shrunk geometric mean** by segment × opportunity
type — averaged in log space, then pulled toward the global value by `n/(n+20)` so a
3-deal group doesn't invent its own number. Imputed values span $3.0k (start-up new)
to $52.6k (enterprise new). Quote a range with coverage stated, never a point.

**`segment=2k` is real but unscoreable.** 1 win in 111 closed deals, yet 10% of open
pipeline. Not a data-entry artifact — 302 deals across 139 creation dates since
2025-02-17, $150k median ACV. It's *young*: 185 of 302 still open, so the only
resolved 2k deals are the fast losses. Its 0.90% win rate is censoring, not a rate.

**541 wins is the binding constraint**, not model choice. Deeper models won't help;
more labelled mid-flight examples will.

**Splitting by `opportunity_type` was tried and is worse.** Renewal wins at 28.4% vs
3.8% for new, so two models looks obviously right — but training on new only costs
PR-AUC 0.5781 → 0.5536, 4× the seed spread. `opportunity_type` is already a feature,
so the tree carves that branch itself where it pays, and pooling keeps the 126
renewal/upsell wins feeding every *other* split. Revisit at ~300 renewal wins.

**`has_champion` direction is real, magnitude isn't.** Won deals have one 81.7% of
the time vs 39.3% for lost — partly because champions help, partly because reps
backfill the field on deals they're already closing.

**`poc` means "POC recorded", not "a POC ran."** It's a checkbox with no unset
state (362 True / 12,601 False / 0 null), and 165 deals have POC *notes* written
with the box still False. The True side carries real signal; the False side
conflates "no POC" with "nobody filled it in."

**`segment_unknown` and `source_unknown` scoring as drivers means the model is
partly learning CRM hygiene** — deals with blank fields lose more often. Real
signal, but it's a data-completeness signal, not deal health. Name it as such.

**We cannot yet measure the baseline that decides ship/no-ship** — today's
stage-constant × amount forecast. On closed deals the stage *is* the label
(8 = won, 9 = lost), so the incumbent scores a meaningless 1.0. Needs point-in-time
history.

**This model is single-tenant.** It learns one company's patterns. Scoring a
*different* business needs multiple orgs in training plus tenant-relative features
and a per-tenant calibration layer — see `framework.md` §4c. Ranking probably
transfers; base rates definitely don't.

**Four things unlock from one change** — start snapshotting `dim_opportunity`
daily. That gives point-in-time training rows (fixing the skew above), stage as a
feature, close-date push count, champion-identified date, and honest backtesting.
`fact_opportunity`'s changelog only starts 2026-05-01 while deals go back to
2023-01, so none of it is recoverable retroactively.